# Demographics (`demo`) harmonization investigation

**Goal:** check whether the 33 columns shared across all 3 raw-domain trials (per
`column_comparison.ipynb`) actually mean the same thing and are coded the same way — not just
present under the same name.

**Findings:**
1. **Core baseline covariates are clean** — `AGE`, `SEX`/`SEXCD`, `RACCAT`/`RACCATCD`,
   `B_ECOGI`/`BECOGICD`, `DIAGTYPE`/`DIAGTYCD`, `B_WEIGHT`/`B_HEIGHT`/`B_BSA` use identical
   coding schemes and units across all 3 trials. Safe to harmonize with a straight rename.
2. **`TRT`/`TRTCD` (treatment arm) is *not* directly harmonizable** — the codes are trial-specific
   integers with trial-specific meanings (e.g. `TRTCD 11/12` in PACCE vs `40/41` in FOLFIRI).
   Needs a derived binary/categorical field (e.g. `on_panitumumab`) built per-trial in the staging
   layer, not a passthrough rename.
3. **`KRAS`/`KRASCD` is missing entirely from `demo` in the FOLFOX/PRIME raw trial** — surprising,
   since PRIME is one of the trials famous for reporting KRAS status. Recovered: it's in that
   trial's separate ADaM export (`NCT00364013_panitumumab_folfox_ADaM/biomark_pds2019.sas7bdat`),
   stored as a name/value pair (`BMMTNM1 == 'KRAS exon 2 (c12/13)'`, result in `BMMTR1`) rather
   than a flat column — needs a cross-file join on `SUBJID`, not a rename.

**Recommendation:** harmonized `demo`-derived staging columns = `age`, `sex`, `race_category`,
`ecog_baseline`, `diagnosis_type`, `weight_kg`, `height_cm`, `bsa_m2` (straight renames) +
`on_panitumumab` (derived per-trial from `TRT`/`TRTCD`) + `kras_status` (straight rename for
PACCE/FOLFIRI, cross-file join against the ADaM `biomark` export for FOLFOX/PRIME).

In [1]:
import pyreadstat
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)

DATA_DIR = Path('..') / '..' / 'Data' / 'raw'

TRIALS = {
    'PACCE (NCT00115225)': 'NCT00115225_PACCE_bev_panitumumab_raw',
    'FOLFIRI (NCT00339183)': 'NCT00339183_panitumumab_folfiri_raw',
    'FOLFOX/PRIME (NCT00364013)': 'NCT00364013_panitumumab_folfox_raw',
}

## 1. Core baseline covariates — value comparison

**What these fields mean:**
- **`AGE`** — age in years at screening.
- **`SEX`/`SEXCD`** — sex, text/code pair (`M`/`F`).
- **`RACCAT`/`RACCATCD`** — race category, text/code pair (`1`=White, `2`=Black, `88`=Other —
  confirmed the same code numbers mean the same categories across trials, not just similar labels).
- **`B_ECOGI`/`BECOGICD`** — baseline ECOG performance status (the standard 0-4 oncology scale for
  how much a patient's daily activity is limited by disease; `0`=fully active, `1`=symptomatic but
  ambulatory, `2`=in bed <50% of the time, etc.) — collected via IVRS (interactive voice response
  system) at baseline.
- **`DIAGTYPE`/`DIAGTYCD`** — primary tumor diagnosis (Colon vs Rectal), same numeric codes
  (`1251`/`1252`) across all 3 trials.
- **`B_WEIGHT`/`B_HEIGHT`/`B_BSA`** — baseline weight (kg), height (cm), body surface area (m²,
  derived from height+weight — relevant since chemo dosing is often BSA-based).

In [2]:
key_cols = ['AGE', 'SEX', 'SEXCD', 'RACCAT', 'RACCATCD', 'B_ECOGI', 'BECOGICD',
            'DIAGTYPE', 'DIAGTYCD', 'B_WEIGHT', 'B_HEIGHT', 'B_BSA']

for label, folder in TRIALS.items():
    path = DATA_DIR / folder / 'demo.sas7bdat'
    df, meta = pyreadstat.read_sas7bdat(str(path))
    label_lookup = dict(zip(meta.column_names, meta.column_labels))
    print(f'===== {label} — demo, n={len(df)} =====')
    for col in key_cols:
        lbl = label_lookup.get(col, '')
        if df[col].nunique(dropna=False) <= 12:
            print(f'-- {col} ({lbl}):')
            print(df[col].value_counts(dropna=False))
        else:
            print(f'-- {col} ({lbl}): numeric, describe:')
            print(df[col].describe())
        print()
    print()

===== PACCE (NCT00115225) — demo, n=842 =====
-- AGE (Age in Years at Screening): numeric, describe:
count    842.000000
mean      60.710214
std       12.041017
min       22.000000
25%       53.000000
50%       61.000000
75%       70.000000
max       89.000000
Name: AGE, dtype: float64

-- SEX (Sex):
SEX
Male      480
Female    362
Name: count, dtype: int64

-- SEXCD (Sex Code):
SEXCD
M    480
F    362
Name: count, dtype: int64

-- RACCAT (Race Category):
RACCAT
White or Caucasian           683
Black or African American     91
Other                         68
Name: count, dtype: int64

-- RACCATCD (Race Category Code):
RACCATCD
1.0     683
2.0      91
88.0     68
Name: count, dtype: int64

-- B_ECOGI (IVRS ECOG Performance Status):
B_ECOGI
Fully active               510
Symptoms but ambulatory    332
Name: count, dtype: int64

-- BECOGICD (IVRS ECOG Performance Status Code):
BECOGICD
0.0    510
1.0    332
Name: count, dtype: int64

-- DIAGTYPE (Primary Tumor Diagnosis):
DIAGTYPE
Colon 

## 2. `TRT`/`TRTCD` — treatment arm is trial-specific, not directly harmonizable

**What these fields mean:** `TRT` (text) / `TRTCD` (numeric code) is the patient's assigned
treatment arm. The problem: each trial has its own arm codes and labels — there's no universal
"code 11 means X" across trials. What's constant is the *concept* (panitumumab-containing arm vs.
control arm), not the code values. The harmonized schema needs a derived flag built per-trial,
mapping each trial's own arm labels to a common `on_panitumumab` boolean (or a richer categorical
if the control-arm chemo backbone matters for modeling, e.g. FOLFOX vs FOLFIRI vs bevacizumab).

In [3]:
for label, folder in TRIALS.items():
    path = DATA_DIR / folder / 'demo.sas7bdat'
    df, meta = pyreadstat.read_sas7bdat(str(path))
    print(f'===== {label} =====')
    print(df[['TRT', 'TRTCD']].value_counts(dropna=False))
    print()

===== PACCE (NCT00115225) =====
TRT                                        TRTCD
panit. plus bevacizumab with chemotherapy  11.0     421
bevacizumab with chemotherapy              12.0     421
Name: count, dtype: int64

===== FOLFIRI (NCT00339183) =====
TRT                    TRTCD
FOLFIRI alone          41.0     475
Panitumumab + FOLFIRI  40.0     471
Name: count, dtype: int64

===== FOLFOX/PRIME (NCT00364013) =====
TRT                   TRTCD
Panitumumab + FOLFOX  42.0     468
FOLFOX alone          43.0     467
Name: count, dtype: int64



## 3. `KRAS`/`KRASCD` — missing from FOLFOX/PRIME's raw `demo`, recovered from its ADaM export

Checked every raw-domain file in `NCT00364013_panitumumab_folfox_raw` for any `KRAS`-named column
— genuinely absent. Checked the ADaM copy of the same trial instead
(`NCT00364013_panitumumab_folfox_ADaM/biomark_pds2019.sas7bdat`), which stores biomarker results
as **name/value pairs across up to 9 numbered "exon" slots** (`BMMTNM1`..`BMMTNM7`, `BMMTNM15`,
`BMMTNM16` hold the biomarker *name*; `BMMTR1`..`BMMTR16` hold the *result*) rather than one flat
column per biomarker. `BMMTNM1 == 'KRAS exon 2 (c12/13)'` is the slot that matches the `KRAS`
field used in the other two trials — same result categories (Wild-type/Mutant/Failure).

In [4]:
# confirm KRAS is genuinely absent from every raw-domain file in FOLFOX/PRIME
folfox_raw = DATA_DIR / 'NCT00364013_panitumumab_folfox_raw'
found_anywhere = False
for f in sorted(folfox_raw.glob('*.sas7bdat')):
    _, meta = pyreadstat.read_sas7bdat(str(f), metadataonly=True)
    matches = [c for c in meta.column_names if 'KRAS' in c.upper()]
    if matches:
        found_anywhere = True
        print(f.name, '->', matches)
print('KRAS found anywhere in raw domain files:', found_anywhere)

KRAS found anywhere in raw domain files: False


In [5]:
# recover it from the ADaM biomark export instead
folfox_adam = DATA_DIR / 'NCT00364013_panitumumab_folfox_ADaM'
biomark, meta = pyreadstat.read_sas7bdat(str(folfox_adam / 'biomark_pds2019.sas7bdat'))

# find which numbered slot holds KRAS exon 2 (c12/13) — the field used in the other 2 trials
for i in [1, 2, 3, 4, 5, 6, 7, 15, 16]:
    name_col = f'BMMTNM{i}'
    if name_col in biomark.columns:
        print(name_col, '->', biomark[name_col].unique())

print()
print("BMMTR1 (KRAS exon 2 result) value_counts:")
print(biomark['BMMTR1'].value_counts(dropna=False))

BMMTNM1 -> <StringArray>
['KRAS exon 2 (c12/13)']
Length: 1, dtype: str
BMMTNM2 -> <StringArray>
['KRAS exon 3 (c61)']
Length: 1, dtype: str
BMMTNM3 -> <StringArray>
['KRAS exon 4 (c117/146)']
Length: 1, dtype: str
BMMTNM4 -> <StringArray>
['NRAS exon 2 (c12/13)']
Length: 1, dtype: str
BMMTNM5 -> <StringArray>
['NRAS exon 3 (c61)']
Length: 1, dtype: str
BMMTNM6 -> <StringArray>
['NRAS exon 4 (c117/146)']
Length: 1, dtype: str
BMMTNM7 -> <StringArray>
['BRAF exon 15 (c600)']
Length: 1, dtype: str
BMMTNM15 -> <StringArray>
['KRAS exon 3 (c59/c61)']
Length: 1, dtype: str
BMMTNM16 -> <StringArray>
['NRAS exon 3 (c59/c61)']
Length: 1, dtype: str

BMMTR1 (KRAS exon 2 result) value_counts:
BMMTR1
Wild-type    514
Mutant       352
              58
Failure       11
Name: count, dtype: int64
